In [1]:
!pip install transformers datasets

In [2]:
import pandas as pd
import kagglehub
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import torch
from google.colab import files
import shutil

In [3]:
path = kagglehub.dataset_download("mgmitesh/sentiment-analysis-dataset")
print("Path to dataset files:", path)

100%|██████████| 15.7M/15.7M [00:02<00:00, 6.01MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/mgmitesh/sentiment-analysis-dataset/versions/2


In [4]:
print(os.listdir(path))

['GoEmotions', 'DailyDialog.csv']


In [5]:
sentiment_dataframe = pd.read_csv(path + "/DailyDialog.csv")
sentiment_dataframe.head()

,text,sentiment
0,I experienced this emotion when my grandfather...,sadness
1,"when I first moved in , I walked everywhere ....",neutral
2,"` Oh ! "" she bleated , her voice high and rath...",anger
3,"However , does the right hon. Gentleman recogn...",fear
4,My boyfriend didn't turn up after promising th...,sadness


In [6]:
sentiment_dataframe.shape

(11327, 2)

In [7]:
sentiment_dataframe = sentiment_dataframe.dropna()

In [8]:
sentiment_dataframe['sentiment'].unique()

array(['sadness', 'neutral', 'anger', 'fear', 'joy'], dtype=object)

In [9]:
sentiments = list(sentiment_dataframe['sentiment'].unique())
sentiment_dictionary = {'sadness':0, 'neutral':1, 'anger':2, 'fear':3, 'joy':4}
type(sentiments)
#keys = [for emotion in emotions]

list

In [10]:
sentiment_digits = []
for sentiment in list(sentiment_dataframe['sentiment']):
  digit = sentiment_dictionary[sentiment]
  sentiment_digits.append(digit)

In [11]:
if len(sentiment_digits)==len(sentiment_dataframe):
  print("Ok")

Ok


In [12]:
sentiment_dataframe["numerized sentiment"] = sentiment_digits
sentiment_dataframe

,text,sentiment,numerized sentiment
0,I experienced this emotion when my grandfather...,sadness,0
1,"when I first moved in , I walked everywhere ....",neutral,1
2,"` Oh ! "" she bleated , her voice high and rath...",anger,2
3,"However , does the right hon. Gentleman recogn...",fear,3
4,My boyfriend didn't turn up after promising th...,sadness,0
...,...,...,...
11322,When I felt alone and without love.,sadness,0
11323,Irina hung up in exasperation .,anger,2
11324,No wonder she was now inconsolable at the pros...,sadness,0
11325,My friend had been telling me about a certain ...,fear,3


In [13]:
train_texts, val_texts, train_labels, val_labels = train_test_split(sentiment_dataframe['text'].tolist(), sentiment_dataframe['numerized sentiment'].tolist(),test_size=0.2, random_state=42)

In [14]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(train_texts, truncation=True, padding=True)
val_encodings = tokenizer(val_texts, truncation=True, padding=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [15]:
class Dataset(torch.utils.data.Dataset):
  def __init__(self, encodings, labels):
    self.encodings = encodings
    self.labels = labels

  def __getitem__(self, idx):
    item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
    item["labels"] = torch.tensor(self.labels[idx])
    return item

  def __len__(self):
    return len(self.labels)

train_dataset = Dataset(train_encodings, train_labels)
val_dataset = Dataset(val_encodings, val_labels)


In [16]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=5
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./logs",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [19]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

Step,Training Loss
500,0.833146
1000,0.619774
1500,0.449159
2000,0.388856
2500,0.310665
3000,0.258089
3500,0.198777
4000,0.124348
4500,0.119721
5000,0.065122


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5665, training_loss=0.3054012086469309, metrics={'train_runtime': 3109.5123, 'train_samples_per_second': 14.57, 'train_steps_per_second': 1.822, 'total_flos': 6449213246249670.0, 'train_loss': 0.3054012086469309, 'epoch': 5.0})

In [20]:
model.save_pretrained("./model")
tokenizer.save_pretrained("./model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./model/tokenizer_config.json', './model/tokenizer.json')

In [21]:
inputs = tokenizer("this movie is amazing", return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}
outputs = model(**inputs)
probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
pred = torch.argmax(probs, dim=1).item()
reverse_map = {v: k for k, v in sentiment_dictionary.items()}
print(reverse_map[pred])

joy


In [22]:
predictions = []
amount_of_texts = len(val_texts)
correct_answers = 0

for text in val_texts:
  inputs = tokenizer(text, return_tensors="pt")
  inputs = {k: v.to(device) for k, v in inputs.items()}
  outputs = model(**inputs)
  probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
  pred = torch.argmax(probs, dim=1).item()
  predictions.append(pred)

for i in range(len(val_texts)):
  if predictions[i] == val_labels[i]:
    correct_answers += 1

accuracy = correct_answers/amount_of_texts*100
print(f'Accuracy: {accuracy:.2f}')

Accuracy: 82.61


In [23]:
shutil.make_archive("./model", "zip", "model")
files.download("model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>